# FAME-DOC — 02. Calibração orientada à decisão e teste independente para KBS

Este notebook implementa a contribuição metodológica adicional proposta para a
versão KBS:

1. calibra os pesos do Hybrid Score **exclusivamente em 2024**;
2. congela os pesos escolhidos;
3. testa FAME-DOC em **2025 sem qualquer recalibração**;
4. compara FAME-DOC com FAME-Fixed (70/20/10) e baselines;
5. executa análise de sensibilidade de 2025 somente como diagnóstico *post hoc*;
6. exporta em CSV **todos os resultados em nível de jogador, rodada, estratégia,
   peso, teste estatístico, estabilidade e auditoria**.

A utilidade usada para calibração é a pontuação observada da equipe **sem bônus
de capitão**, porque escalação e capitão representam decisões distintas.
**Versão v4 KBS:** inclui estudo de ablação out-of-time de 2025 e figuras específicas para discussão dos resultados.


In [ ]:
# ============================================================
# 1. DEPENDÊNCIAS
# ============================================================
from pathlib import Path
from itertools import combinations
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import pulp

from scipy.stats import friedmanchisquare, wilcoxon, spearmanr, kendalltau

SEED = 42
BOOTSTRAP_SEED = 123
rng = np.random.default_rng(SEED)

## 2. Configuração e entradas

In [ ]:
BASE_DIR = Path.cwd()
ROOT = BASE_DIR / "resultados_fame_doc_kbs"
CAL_DIR = ROOT / "01_previsoes_calibracao_2024"
OUT = ROOT / "02_calibracao_e_teste_2025"
OUT.mkdir(parents=True, exist_ok=True)

CAL_CANDIDATES = [
    CAL_DIR / "doc_calibration_predictions_2024_probable.csv",
    BASE_DIR / "doc_calibration_predictions_2024_probable.csv",
]
TEST_CANDIDATES = [
    BASE_DIR / "avaliacao_previsoes_teste_2025_provaveis.csv",
    BASE_DIR / "resultados_discussao" / "tabelas" / "avaliacao_previsoes_teste_2025_provaveis.csv",
    BASE_DIR.parent / "avaliacao_previsoes_teste_2025_provaveis.csv",
]

CAL_CSV = next((p for p in CAL_CANDIDATES if p.exists()), CAL_CANDIDATES[0])
TEST_CSV = next((p for p in TEST_CANDIDATES if p.exists()), TEST_CANDIDATES[0])

REFERENCE_WEIGHTS = (0.70, 0.20, 0.10)
GRID_STEP = 0.05
NEAR_OPTIMAL_TOLERANCE = 0.01
N_BOOTSTRAP = 5000

FORMATION = "4-3-3"
BUDGET_MAX = 250.0
BUDGET_MIN = 0.0
MAX_PER_CLUB = 11  # reproduz o experimento ESWA; restrição não vinculante na prática
TOP_N_PER_POSITION = 80
MIN_GAMES = 0
REQUIRE_PROBABLE = True

print("Calibração:", CAL_CSV.resolve())
print("Teste:", TEST_CSV.resolve())

# Contrato da formação 4-3-3 usada no experimento:
EXPECTED_FIELD_PLAYERS = 11
EXPECTED_COACHES = 1
EXPECTED_TOTAL_SELECTED = EXPECTED_FIELD_PLAYERS + EXPECTED_COACHES


## 3. Leitura, padronização e auditoria de separação temporal

In [ ]:
def load_prediction_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path.resolve()}")
    d = pd.read_csv(path)
    ren = {"rodada": "rodada_target", "preco": "preco_num", "pontos": "pontos_num"}
    for old, new in ren.items():
        if old in d.columns and new not in d.columns:
            d = d.rename(columns={old: new})
    numeric = [
        "temporada", "rodada_target", "posicao_id", "preco_num", "pontos_num",
        "pred_rf", "pred_xgb", "pred_ensemble", "pred_baseline",
        "pred_ajustada_prob", "teto_score", "roi", "std_shrunk",
        "prob_jogar", "status_id", "jogos_num", "media_num",
    ]
    for c in numeric:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    return d

cal = load_prediction_file(CAL_CSV)
test = load_prediction_file(TEST_CSV)

if set(cal["temporada"].dropna().astype(int).unique()) != {2024}:
    raise AssertionError("A base de calibração deve conter exclusivamente 2024.")
if set(test["temporada"].dropna().astype(int).unique()) != {2025}:
    raise AssertionError("A base de teste deve conter exclusivamente 2025.")

ID_CANDIDATES = ["atleta_id", "player_id", "id_atleta", "slug", "apelido", "nome"]
PLAYER_ID_COL = next((c for c in ID_CANDIDATES if c in cal.columns and c in test.columns), None)
if PLAYER_ID_COL is None:
    raise ValueError("É necessário um identificador estável de jogador comum às duas bases.")

temporal_audit = pd.DataFrame([
    {
        "dataset": "Decision calibration",
        "season": 2024,
        "n_rows": len(cal),
        "n_rounds": cal["rodada_target"].nunique(),
        "n_players": cal[PLAYER_ID_COL].nunique(),
        "role": "weight selection",
    },
    {
        "dataset": "Independent test",
        "season": 2025,
        "n_rows": len(test),
        "n_rounds": test["rodada_target"].nunique(),
        "n_players": test[PLAYER_ID_COL].nunique(),
        "role": "final evaluation only",
    },
])
temporal_audit.to_csv(OUT / "kbs_doc_temporal_split_audit.csv", index=False, encoding="utf-8-sig")
display(temporal_audit)

## 4. Componentes decisórios e normalização por rodada

In [ ]:
PROB_PLAY_STATUS_MAP = {2: 0.50, 3: 0.00, 5: 0.00, 6: 0.05, 7: 0.90}
CEILING_VARIABILITY_FACTOR = 0.25

def minmax_por_rodada(df, coluna):
    def scale(s):
        s = pd.to_numeric(s, errors="coerce")
        lo, hi = s.min(), s.max()
        if pd.isna(lo) or pd.isna(hi) or np.isclose(lo, hi):
            return pd.Series(0.0, index=s.index)
        return (s - lo) / (hi - lo)
    return df.groupby(["temporada", "rodada_target"], dropna=False)[coluna].transform(scale)

def ensure_components(d):
    d = d.copy()
    if "prob_jogar" not in d.columns:
        if "status_id" not in d.columns:
            raise ValueError("Faltam prob_jogar e status_id.")
        d["prob_jogar"] = d["status_id"].map(PROB_PLAY_STATUS_MAP).fillna(0.50)

    if "pred_ajustada_prob" not in d.columns:
        d["pred_ajustada_prob"] = d["pred_ensemble"] * d["prob_jogar"]

    if "teto_score" not in d.columns:
        if "std_shrunk" in d.columns:
            d["teto_score"] = d["pred_ajustada_prob"] + CEILING_VARIABILITY_FACTOR * d["std_shrunk"].fillna(0)
        elif "std_5" in d.columns:
            d["teto_score"] = d["pred_ajustada_prob"] + CEILING_VARIABILITY_FACTOR * d["std_5"].fillna(0)
        else:
            raise ValueError("Não é possível reconstruir teto_score.")

    if "roi" not in d.columns:
        d["roi"] = d["pred_ensemble"] / np.sqrt(d["preco_num"].clip(lower=1e-6))

    d["component_expected_norm"] = minmax_por_rodada(d, "pred_ajustada_prob")
    d["component_ceiling_norm"] = minmax_por_rodada(d, "teto_score")
    d["component_economic_norm"] = minmax_por_rodada(d, "roi")
    return d

cal = ensure_components(cal)
test = ensure_components(test)

cal.to_csv(OUT / "kbs_doc_calibration_player_components_2024.csv", index=False, encoding="utf-8-sig")
test.to_csv(OUT / "kbs_doc_test_player_components_2025.csv", index=False, encoding="utf-8-sig")

## 5. Grade completa de pesos

In [ ]:
def generate_weight_grid(step=0.05):
    n = int(round(1 / step))
    grid = []
    for i in range(n + 1):
        for j in range(n + 1 - i):
            k = n - i - j
            grid.append((i / n, j / n, k / n))
    return grid

weight_grid = generate_weight_grid(GRID_STEP)
weight_grid_df = pd.DataFrame(weight_grid, columns=["w_expected", "w_ceiling", "w_economic"])
weight_grid_df["weight_id"] = np.arange(1, len(weight_grid_df) + 1)
weight_grid_df.to_csv(OUT / "kbs_doc_weight_grid.csv", index=False, encoding="utf-8-sig")

if np.isclose(GRID_STEP, 0.05):
    assert len(weight_grid) == 231
print("Combinações:", len(weight_grid))

## 6. Otimização inteira e avaliação de uma estratégia

In [ ]:
ESQUEMAS = {
    "3-4-3": {"goleiro": 1, "lateral": 0, "zagueiro": 3, "meia": 4, "atacante": 3, "tecnico": 1},
    "4-3-3": {"goleiro": 1, "lateral": 2, "zagueiro": 2, "meia": 3, "atacante": 3, "tecnico": 1},
    "4-4-2": {"goleiro": 1, "lateral": 2, "zagueiro": 2, "meia": 4, "atacante": 2, "tecnico": 1},
    "3-5-2": {"goleiro": 1, "lateral": 0, "zagueiro": 3, "meia": 5, "atacante": 2, "tecnico": 1},
    "5-3-2": {"goleiro": 1, "lateral": 2, "zagueiro": 3, "meia": 3, "atacante": 2, "tecnico": 1},
    "5-4-1": {"goleiro": 1, "lateral": 2, "zagueiro": 3, "meia": 4, "atacante": 1, "tecnico": 1},
}
MAPA_POSICOES = {"goleiro": 1, "lateral": 2, "zagueiro": 3, "meia": 4, "atacante": 5, "tecnico": 6}

def selecionar_time(df_rodada, score_col):
    d = df_rodada.copy()
    if REQUIRE_PROBABLE and "status_id" in d.columns:
        d = d[d["status_id"] == 7].copy()
    if MIN_GAMES is not None and "jogos_num" in d.columns:
        d = d[d["jogos_num"].fillna(0) >= MIN_GAMES].copy()
    d = d[d["preco_num"].fillna(0) > 0].copy()

    parts = []
    for pos_name, qty in ESQUEMAS[FORMATION].items():
        if qty == 0:
            continue
        pos_id = MAPA_POSICOES[pos_name]
        part = (
            d[d["posicao_id"] == pos_id]
            .dropna(subset=[score_col, "preco_num", "pontos_num"])
            .sort_values([score_col, "preco_num", PLAYER_ID_COL], ascending=[False, True, True])
            .head(TOP_N_PER_POSITION)
            .copy()
        )
        if len(part) < qty:
            return None, f"Candidatos insuficientes em {pos_name}: {len(part)} < {qty}"
        parts.append(part)

    candidates = pd.concat(parts, ignore_index=True)
    problem = pulp.LpProblem("FAME_DOC", pulp.LpMaximize)
    x = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in candidates.index}

    problem += pulp.lpSum(x[i] * float(candidates.loc[i, score_col]) for i in candidates.index)
    cost = pulp.lpSum(x[i] * float(candidates.loc[i, "preco_num"]) for i in candidates.index)
    problem += cost <= BUDGET_MAX
    if BUDGET_MIN > 0:
        problem += cost >= BUDGET_MIN

    for pos_name, qty in ESQUEMAS[FORMATION].items():
        if qty == 0:
            continue
        pos_id = MAPA_POSICOES[pos_name]
        problem += pulp.lpSum(
            x[i] for i in candidates.index if candidates.loc[i, "posicao_id"] == pos_id
        ) == qty

    if "clube_id" in candidates.columns:
        for club in candidates["clube_id"].dropna().unique():
            problem += pulp.lpSum(
                x[i] for i in candidates.index if candidates.loc[i, "clube_id"] == club
            ) <= MAX_PER_CLUB

    status = problem.solve(pulp.PULP_CBC_CMD(msg=False))
    if pulp.LpStatus[status] != "Optimal":
        return None, f"Solver: {pulp.LpStatus[status]}"

    selected = [i for i in candidates.index if x[i].value() and x[i].value() > 0.5]
    team = candidates.loc[selected].copy()

    # Contrato estrutural: 11 jogadores de campo + 1 técnico = 12 entidades selecionadas.
    n_coaches = int((team["posicao_id"] == MAPA_POSICOES["tecnico"]).sum())
    n_field = int(len(team) - n_coaches)
    if FORMATION == "4-3-3":
        if len(team) != EXPECTED_TOTAL_SELECTED or n_field != EXPECTED_FIELD_PLAYERS or n_coaches != EXPECTED_COACHES:
            raise AssertionError(
                f"Formação inconsistente: total={len(team)}, jogadores={n_field}, técnicos={n_coaches}"
            )

    return team, None

def lineup_signature(team):
    return "|".join(sorted(team[PLAYER_ID_COL].astype(str).tolist()))

def evaluate_score_column(df, score_col, strategy_name, extra_metadata=None):
    rows, player_rows, failures = [], [], []
    extra_metadata = extra_metadata or {}

    for (season, round_target), group in df.groupby(["temporada", "rodada_target"], dropna=False):
        team, reason = selecionar_time(group, score_col)
        if team is None:
            failures.append({
                "temporada": season, "rodada_target": round_target,
                "strategy": strategy_name, "reason": reason, **extra_metadata
            })
            continue

        utility = float(team["pontos_num"].sum())  # SEM capitão: objetivo de calibração
        total_cost = float(team["preco_num"].sum())
        objective_value = float(team[score_col].sum())

        rows.append({
            "temporada": season,
            "rodada_target": round_target,
            "strategy": strategy_name,
            "score_column": score_col,
            "operational_utility_no_captain": utility,
            "objective_value": objective_value,
            "cost_total": total_cost,
            "budget_remaining": BUDGET_MAX - total_cost,
            "n_selected": len(team),
            "n_field_players": int((team["posicao_id"] != MAPA_POSICOES["tecnico"]).sum()),
            "n_coaches": int((team["posicao_id"] == MAPA_POSICOES["tecnico"]).sum()),
            "max_selected_from_same_club": (
                int(team["clube_id"].value_counts().max()) if "clube_id" in team.columns and not team.empty else np.nan
            ),
            "club_limit_parameter": MAX_PER_CLUB,
            "club_limit_binding": (
                bool(team["clube_id"].value_counts().max() >= MAX_PER_CLUB)
                if "clube_id" in team.columns and not team.empty else False
            ),
            "n_clubs": team["clube_id"].nunique() if "clube_id" in team.columns else np.nan,
            "lineup_signature": lineup_signature(team),
            **extra_metadata,
        })

        det = team.copy()
        det["temporada_backtest"] = season
        det["rodada_backtest"] = round_target
        det["strategy"] = strategy_name
        det["score_column"] = score_col
        det["selected_objective_value"] = det[score_col]
        for k, v in extra_metadata.items():
            det[k] = v
        player_rows.append(det)

    return (
        pd.DataFrame(rows),
        pd.concat(player_rows, ignore_index=True) if player_rows else pd.DataFrame(),
        pd.DataFrame(failures),
    )


## 6A. Contrato de otimização auditado

A implementação original usada nos experimentos da ESWA seleciona, na formação
4-3-3, **12 entidades pontuáveis: 11 jogadores + 1 técnico**. Portanto, a versão
KBS não deve usar a equação `Σxᵢ = 11` sobre todo o universo de candidatos.

A formulação correta deve distinguir jogadores e técnico, ou simplesmente impor
as igualdades posicionais, cuja soma é 12.

Também foi constatado que `MAX_PER_CLUB = 11` reproduz a implementação original,
mas torna a restrição por clube praticamente não vinculante. A versão KBS deve
descrever esse parâmetro como configuração experimental, sem afirmar que ele foi
um limite oficial ativo, salvo se isso for posteriormente documentado.


In [ ]:

formation_contract = pd.DataFrame([
    {"position": "Goalkeeper", "posicao_id": 1, "required": 1},
    {"position": "Full-back", "posicao_id": 2, "required": 2},
    {"position": "Centre-back", "posicao_id": 3, "required": 2},
    {"position": "Midfielder", "posicao_id": 4, "required": 3},
    {"position": "Forward", "posicao_id": 5, "required": 3},
    {"position": "Coach", "posicao_id": 6, "required": 1},
])
formation_contract["entity_type"] = np.where(
    formation_contract["position"].eq("Coach"), "coach", "field_player"
)

optimization_contract = pd.DataFrame([
    {"item": "formation", "value": FORMATION},
    {"item": "field_players", "value": EXPECTED_FIELD_PLAYERS},
    {"item": "coaches", "value": EXPECTED_COACHES},
    {"item": "total_selected_entities", "value": EXPECTED_TOTAL_SELECTED},
    {"item": "budget_max", "value": BUDGET_MAX},
    {"item": "club_limit_parameter", "value": MAX_PER_CLUB},
    {"item": "club_limit_interpretation", "value": "implemented; expected to be nonbinding with value 11"},
    {"item": "captain_in_calibration_objective", "value": False},
])

formation_contract.to_csv(OUT / "kbs_doc_formation_contract.csv", index=False, encoding="utf-8-sig")
optimization_contract.to_csv(OUT / "kbs_doc_optimization_contract.csv", index=False, encoding="utf-8-sig")

display(formation_contract)
display(optimization_contract)



## 6B. Auditoria de consistência para reescrita KBS

Este arquivo não altera resultados; ele documenta pontos que precisam ser
corrigidos na redação do manuscrito KBS para refletir a implementação real.


In [ ]:

code_paper_audit = pd.DataFrame([
    {
        "topic": "team size",
        "eswa_text_or_claim": "exactly eleven players selected over the complete candidate set",
        "implemented_behavior": "11 field players plus 1 coach = 12 selected entities",
        "kbs_action": "rewrite mathematical formulation and all prose referring to team size",
        "severity": "critical"
    },
    {
        "topic": "hyperparameter tuning",
        "eswa_text_or_claim": "hyperparameter optimization independently for each position",
        "implemented_behavior": "fixed pre-specified RF and XGBoost settings shared across positions",
        "kbs_action": "remove tuning claim; report exact fixed configurations",
        "severity": "critical"
    },
    {
        "topic": "club constraint",
        "eswa_text_or_claim": "active club-composition constraint",
        "implemented_behavior": "constraint coded with MAX_PER_CLUB=11; effectively nonbinding in main experiment",
        "kbs_action": "describe as implemented experimental parameter; do not call it an active official constraint without documentation",
        "severity": "major"
    },
    {
        "topic": "captain objective",
        "eswa_text_or_claim": "lineup score includes captain analysis",
        "implemented_behavior": "DOC weight calibration explicitly excludes captain bonus",
        "kbs_action": "separate lineup utility from captain decision",
        "severity": "methodological clarification"
    },
])
code_paper_audit.to_csv(
    OUT / "kbs_doc_code_paper_consistency_audit.csv",
    index=False, encoding="utf-8-sig"
)
display(code_paper_audit)


## 7. Calibração orientada à decisão em 2024

In [ ]:
cal_round_all, cal_players_all, cal_fail_all = [], [], []

for idx, (wp, wt, wr) in enumerate(weight_grid, start=1):
    d = cal.copy()
    d["score_doc_candidate"] = (
        wp * d["component_expected_norm"]
        + wt * d["component_ceiling_norm"]
        + wr * d["component_economic_norm"]
    )
    metadata = {"w_expected": wp, "w_ceiling": wt, "w_economic": wr}
    rr, pp, ff = evaluate_score_column(
        d, "score_doc_candidate", "DOC calibration candidate", metadata
    )
    cal_round_all.append(rr)
    cal_players_all.append(pp)
    if not ff.empty:
        cal_fail_all.append(ff)

    if idx % 25 == 0 or idx == len(weight_grid):
        print(f"{idx}/{len(weight_grid)} pesos calibrados.")

cal_results_round = pd.concat(cal_round_all, ignore_index=True)
cal_results_players = pd.concat(cal_players_all, ignore_index=True)
cal_failures = pd.concat(cal_fail_all, ignore_index=True) if cal_fail_all else pd.DataFrame()

weight_cols = ["w_expected", "w_ceiling", "w_economic"]
cal_summary = (
    cal_results_round.groupby(weight_cols, as_index=False)
    .agg(
        n_rounds=("operational_utility_no_captain", "size"),
        mean_utility=("operational_utility_no_captain", "mean"),
        median_utility=("operational_utility_no_captain", "median"),
        sd_utility=("operational_utility_no_captain", "std"),
        cumulative_utility=("operational_utility_no_captain", "sum"),
        min_utility=("operational_utility_no_captain", "min"),
        max_utility=("operational_utility_no_captain", "max"),
        mean_cost=("cost_total", "mean"),
        mean_budget_remaining=("budget_remaining", "mean"),
    )
)

best_mean = cal_summary["mean_utility"].max()
cal_summary["within_1pct_best"] = cal_summary["mean_utility"] >= (1 - NEAR_OPTIMAL_TOLERANCE) * best_mean
cal_summary["rank_mean"] = cal_summary["mean_utility"].rank(ascending=False, method="min").astype(int)
cal_summary = cal_summary.sort_values(["mean_utility", "sd_utility"], ascending=[False, True]).reset_index(drop=True)

selected = cal_summary.iloc[0]
DOC_WEIGHTS = (
    float(selected["w_expected"]),
    float(selected["w_ceiling"]),
    float(selected["w_economic"]),
)

selected_weights = pd.DataFrame([{
    "selection_dataset": "2024 decision calibration",
    "selection_objective": "mean operational utility without captain",
    "w_expected": DOC_WEIGHTS[0],
    "w_ceiling": DOC_WEIGHTS[1],
    "w_economic": DOC_WEIGHTS[2],
    "mean_utility_calibration": float(selected["mean_utility"]),
    "sd_utility_calibration": float(selected["sd_utility"]),
    "rank_calibration": int(selected["rank_mean"]),
    "weights_frozen_before_2025": True,
}])

# Exporta imediatamente os resultados completos da calibração.
cal_results_round.to_csv(OUT / "kbs_doc_calibration_results_by_round_all_weights.csv", index=False, encoding="utf-8-sig")
cal_results_players.to_csv(OUT / "kbs_doc_calibration_selected_players_all_weights.csv", index=False, encoding="utf-8-sig")
cal_summary.to_csv(OUT / "kbs_doc_cal_summary_all_weights.csv", index=False, encoding="utf-8-sig")
cal_summary[cal_summary["within_1pct_best"]].to_csv(OUT / "kbs_doc_calibration_near_optimal_region.csv", index=False, encoding="utf-8-sig")
selected_weights.to_csv(OUT / "kbs_doc_selected_weights_frozen.csv", index=False, encoding="utf-8-sig")
if not cal_failures.empty:
    cal_failures.to_csv(OUT / "kbs_doc_calibration_failures.csv", index=False, encoding="utf-8-sig")

print("Pesos DOC congelados:", DOC_WEIGHTS)
display(selected_weights)


## 8. Teste independente de 2025 — FAME-DOC, FAME-Fixed e baselines

In [ ]:
# Scores Fixed e DOC
test_eval = test.copy()
wp, wt, wr = REFERENCE_WEIGHTS
test_eval["score_fame_fixed"] = (
    wp * test_eval["component_expected_norm"]
    + wt * test_eval["component_ceiling_norm"]
    + wr * test_eval["component_economic_norm"]
)
wp, wt, wr = DOC_WEIGHTS
test_eval["score_fame_doc"] = (
    wp * test_eval["component_expected_norm"]
    + wt * test_eval["component_ceiling_norm"]
    + wr * test_eval["component_economic_norm"]
)

strategies = {
    "FAME-DOC": "score_fame_doc",
    "FAME-Fixed 70/20/10": "score_fame_fixed",
    "Pure prediction": "pred_ensemble",
    "Availability-adjusted prediction": "pred_ajustada_prob",
    "Ceiling score": "teto_score",
    "Economic efficiency": "roi",
    "Historical mean": "media_num",
    "Market value": "preco_num",
}

test_round_parts, test_player_parts, test_fail_parts = [], [], []
for strategy, col in strategies.items():
    if col not in test_eval.columns:
        print("Aviso: estratégia ignorada por coluna ausente:", strategy, col)
        continue
    rr, pp, ff = evaluate_score_column(test_eval, col, strategy)
    test_round_parts.append(rr)
    test_player_parts.append(pp)
    if not ff.empty:
        test_fail_parts.append(ff)

test_round = pd.concat(test_round_parts, ignore_index=True)
test_players = pd.concat(test_player_parts, ignore_index=True)
test_failures = pd.concat(test_fail_parts, ignore_index=True) if test_fail_parts else pd.DataFrame()

test_summary = (
    test_round.groupby("strategy", as_index=False)
    .agg(
        n_rounds=("operational_utility_no_captain", "size"),
        mean_utility=("operational_utility_no_captain", "mean"),
        median_utility=("operational_utility_no_captain", "median"),
        sd_utility=("operational_utility_no_captain", "std"),
        cumulative_utility=("operational_utility_no_captain", "sum"),
        min_utility=("operational_utility_no_captain", "min"),
        max_utility=("operational_utility_no_captain", "max"),
        mean_cost=("cost_total", "mean"),
        mean_budget_remaining=("budget_remaining", "mean"),
    )
    .sort_values("mean_utility", ascending=False)
)

pred_mean = test_summary.loc[test_summary["strategy"] == "Pure prediction", "mean_utility"]
if not pred_mean.empty:
    test_summary["delta_mean_vs_pure_prediction"] = test_summary["mean_utility"] - float(pred_mean.iloc[0])

test_eval.to_csv(OUT / "kbs_doc_test_player_scores_2025.csv", index=False, encoding="utf-8-sig")
test_round.to_csv(OUT / "kbs_doc_test_results_by_round.csv", index=False, encoding="utf-8-sig")
test_players.to_csv(OUT / "kbs_doc_test_selected_players_by_strategy.csv", index=False, encoding="utf-8-sig")
test_summary.to_csv(OUT / "kbs_doc_test_strategy_summary.csv", index=False, encoding="utf-8-sig")
if not test_failures.empty:
    test_failures.to_csv(OUT / "kbs_doc_test_failures.csv", index=False, encoding="utf-8-sig")

display(test_summary)

## 9. Bootstrap, Friedman e Wilcoxon pareado com correção de Holm

In [ ]:
def bootstrap_ci(x, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    rg = np.random.default_rng(seed)
    if len(x) == 0:
        return np.nan, np.nan
    means = rg.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]))

boot_rows = []
for i, (strategy, g) in enumerate(test_round.groupby("strategy")):
    lo, hi = bootstrap_ci(g["operational_utility_no_captain"], seed=BOOTSTRAP_SEED + i)
    boot_rows.append({
        "strategy": strategy,
        "mean": g["operational_utility_no_captain"].mean(),
        "ci95_low": lo,
        "ci95_high": hi,
        "n_rounds": len(g),
    })
bootstrap_table = pd.DataFrame(boot_rows)
bootstrap_table.to_csv(OUT / "kbs_doc_test_bootstrap_ci.csv", index=False, encoding="utf-8-sig")

pivot = test_round.pivot_table(
    index=["temporada", "rodada_target"],
    columns="strategy",
    values="operational_utility_no_captain",
    aggfunc="first",
).dropna()

friedman_table = pd.DataFrame()
if pivot.shape[1] >= 3 and len(pivot) > 0:
    stat, p = friedmanchisquare(*[pivot[c].values for c in pivot.columns])
    friedman_table = pd.DataFrame([{
        "test": "Friedman",
        "statistic": stat,
        "p_value": p,
        "n_rounds_complete": len(pivot),
        "n_strategies": pivot.shape[1],
    }])
friedman_table.to_csv(OUT / "kbs_doc_test_friedman.csv", index=False, encoding="utf-8-sig")

def holm_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running = max(running, value)
        adjusted[idx] = min(running, 1.0)
    return adjusted

pair_rows = []
for a, b in combinations(pivot.columns, 2):
    diff = pivot[a] - pivot[b]
    try:
        stat, p = wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
    except ValueError:
        stat, p = 0.0, 1.0
    pair_rows.append({
        "strategy_a": a,
        "strategy_b": b,
        "n_rounds": len(diff),
        "mean_a": pivot[a].mean(),
        "mean_b": pivot[b].mean(),
        "mean_diff_a_minus_b": diff.mean(),
        "median_diff_a_minus_b": diff.median(),
        "wins_a_pct": 100 * (diff > 0).mean(),
        "ties_pct": 100 * (diff == 0).mean(),
        "wilcoxon_stat": stat,
        "p_raw": p,
    })

pairwise = pd.DataFrame(pair_rows)
if not pairwise.empty:
    pairwise["p_holm"] = holm_adjust(pairwise["p_raw"].to_numpy())
    pairwise["significant_5pct"] = pairwise["p_holm"] < 0.05
pairwise.to_csv(OUT / "kbs_doc_test_pairwise_wilcoxon_holm.csv", index=False, encoding="utf-8-sig")

display(friedman_table)
display(pairwise.sort_values("p_holm").head(20) if not pairwise.empty else pairwise)

## 10. Métricas formais de ranking — incluindo FAME-DOC

In [ ]:
def _dcg(rel):
    rel = np.asarray(rel, dtype=float)
    if rel.size == 0:
        return np.nan
    return float(np.sum(rel / np.log2(np.arange(2, len(rel) + 2))))

def ranking_metrics_round(group, criterion, k_requested, y_true="pontos_num"):
    d = group.dropna(subset=[criterion, y_true]).drop_duplicates(PLAYER_ID_COL, keep="last").copy()
    if len(d) < 2:
        return None

    ranking_pred = d.sort_values([criterion, "preco_num", PLAYER_ID_COL], ascending=[False, True, True])
    ranking_real = d.sort_values([y_true, PLAYER_ID_COL], ascending=[False, True])

    rp = d[criterion].rank(method="average", ascending=False)
    rr = d[y_true].rank(method="average", ascending=False)
    sp = spearmanr(rp, rr, nan_policy="omit").statistic
    kt = kendalltau(rp, rr, nan_policy="omit").statistic

    k = min(k_requested, len(d))
    ids_pred = ranking_pred.head(k)[PLAYER_ID_COL].tolist()
    ids_real = set(ranking_real.head(k)[PLAYER_ID_COL].tolist())
    overlap = len(set(ids_pred) & ids_real)

    min_y = float(d[y_true].min())
    shift = -min_y if min_y < 0 else 0.0
    relevance = (d.set_index(PLAYER_ID_COL)[y_true] + shift).clip(lower=0.0)
    dcg = _dcg(relevance.reindex(ids_pred).fillna(0).to_numpy())
    idcg = _dcg(np.sort(relevance.to_numpy())[::-1][:k])

    return {
        "k": k_requested,
        "k_effective": k,
        "n_candidates": len(d),
        "precision_at_k": overlap / k,
        "recall_at_k": overlap / len(ids_real) if ids_real else np.nan,
        "ndcg_at_k": dcg / idcg if idcg and idcg > 0 else np.nan,
        "spearman": sp,
        "kendall": kt,
        "mean_observed_topk_predicted": ranking_pred.head(k)[y_true].mean(),
        "sum_observed_topk_predicted": ranking_pred.head(k)[y_true].sum(),
        "overlap_topk": overlap,
    }

ranking_criteria = {
    "FAME-DOC": "score_fame_doc",
    "FAME-Fixed 70/20/10": "score_fame_fixed",
    "Pure prediction": "pred_ensemble",
    "Availability-adjusted prediction": "pred_ajustada_prob",
    "Ceiling score": "teto_score",
    "Economic efficiency": "roi",
    "Historical mean": "media_num",
    "Market value": "preco_num",
}

rank_rows = []
for strategy, col in ranking_criteria.items():
    if col not in test_eval.columns:
        continue
    for (season, round_target), g in test_eval.groupby(["temporada", "rodada_target"]):
        for k in (5, 10, 20):
            r = ranking_metrics_round(g, col, k)
            if r:
                r.update({
                    "temporada": season,
                    "rodada_target": round_target,
                    "strategy": strategy,
                    "criterion": col,
                })
                rank_rows.append(r)

ranking_round = pd.DataFrame(rank_rows)
ranking_summary = (
    ranking_round.groupby(["strategy", "criterion", "k"], as_index=False)
    .agg(
        n_rounds=("ndcg_at_k", "count"),
        precision_mean=("precision_at_k", "mean"),
        precision_sd=("precision_at_k", "std"),
        recall_mean=("recall_at_k", "mean"),
        recall_sd=("recall_at_k", "std"),
        ndcg_mean=("ndcg_at_k", "mean"),
        ndcg_sd=("ndcg_at_k", "std"),
        spearman_mean=("spearman", "mean"),
        spearman_sd=("spearman", "std"),
        kendall_mean=("kendall", "mean"),
        kendall_sd=("kendall", "std"),
        mean_observed_topk=("mean_observed_topk_predicted", "mean"),
        sum_observed_topk=("sum_observed_topk_predicted", "mean"),
    )
)

ranking_round.to_csv(OUT / "kbs_doc_ranking_metrics_by_round.csv", index=False, encoding="utf-8-sig")
ranking_summary.to_csv(OUT / "kbs_doc_ranking_metrics_summary.csv", index=False, encoding="utf-8-sig")
display(ranking_summary)

## 11. Sensibilidade independente em 2025 — diagnóstico, não seleção

In [ ]:
test_sens_round_parts, test_sens_player_parts, test_sens_fail_parts = [], [], []

for idx, (wp, wt, wr) in enumerate(weight_grid, start=1):
    d = test.copy()
    d["score_posthoc_sensitivity"] = (
        wp * d["component_expected_norm"]
        + wt * d["component_ceiling_norm"]
        + wr * d["component_economic_norm"]
    )
    metadata = {"w_expected": wp, "w_ceiling": wt, "w_economic": wr}
    rr, pp, ff = evaluate_score_column(
        d, "score_posthoc_sensitivity", "2025 post-hoc sensitivity", metadata
    )
    test_sens_round_parts.append(rr)
    test_sens_player_parts.append(pp)
    if not ff.empty:
        test_sens_fail_parts.append(ff)

test_sens_round = pd.concat(test_sens_round_parts, ignore_index=True)
test_sens_players = pd.concat(test_sens_player_parts, ignore_index=True)
test_sens_failures = (
    pd.concat(test_sens_fail_parts, ignore_index=True)
    if test_sens_fail_parts else pd.DataFrame()
)

test_sens_summary = (
    test_sens_round.groupby(weight_cols, as_index=False)
    .agg(
        n_rounds=("operational_utility_no_captain", "size"),
        mean_utility=("operational_utility_no_captain", "mean"),
        median_utility=("operational_utility_no_captain", "median"),
        sd_utility=("operational_utility_no_captain", "std"),
        cumulative_utility=("operational_utility_no_captain", "sum"),
    )
    .sort_values(["mean_utility", "sd_utility"], ascending=[False, True])
    .reset_index(drop=True)
)
test_sens_summary["rank_mean"] = np.arange(1, len(test_sens_summary) + 1)
test_best = test_sens_summary.iloc[0]
TEST_RETROSPECTIVE_BEST = (
    float(test_best["w_expected"]),
    float(test_best["w_ceiling"]),
    float(test_best["w_economic"]),
)

test_sens_round.to_csv(OUT / "kbs_doc_test_2025_posthoc_sensitivity_by_round_all_weights.csv", index=False, encoding="utf-8-sig")
test_sens_players.to_csv(OUT / "kbs_doc_test_2025_posthoc_sensitivity_players_all_weights.csv", index=False, encoding="utf-8-sig")
test_sens_summary.to_csv(OUT / "kbs_doc_test_2025_posthoc_sensitivity_summary_all_weights.csv", index=False, encoding="utf-8-sig")
if not test_sens_failures.empty:
    test_sens_failures.to_csv(OUT / "kbs_doc_test_2025_posthoc_sensitivity_failures.csv", index=False, encoding="utf-8-sig")

print("Melhor retrospectivo 2025 (NÃO usado para seleção):", TEST_RETROSPECTIVE_BEST)

## 12. Estabilidade temporal dos pesos e das escalações

In [ ]:
doc = np.asarray(DOC_WEIGHTS, dtype=float)
fixed = np.asarray(REFERENCE_WEIGHTS, dtype=float)
best25 = np.asarray(TEST_RETROSPECTIVE_BEST, dtype=float)

weight_stability = pd.DataFrame([
    {
        "comparison": "DOC 2024 vs Fixed 70/20/10",
        "l1_distance": float(np.abs(doc - fixed).sum()),
        "l2_distance": float(np.sqrt(((doc - fixed) ** 2).sum())),
        "w_expected_a": doc[0], "w_ceiling_a": doc[1], "w_economic_a": doc[2],
        "w_expected_b": fixed[0], "w_ceiling_b": fixed[1], "w_economic_b": fixed[2],
    },
    {
        "comparison": "DOC 2024 vs retrospective best 2025",
        "l1_distance": float(np.abs(doc - best25).sum()),
        "l2_distance": float(np.sqrt(((doc - best25) ** 2).sum())),
        "w_expected_a": doc[0], "w_ceiling_a": doc[1], "w_economic_a": doc[2],
        "w_expected_b": best25[0], "w_ceiling_b": best25[1], "w_economic_b": best25[2],
    },
])
weight_stability.to_csv(OUT / "kbs_doc_temporal_weight_stability.csv", index=False, encoding="utf-8-sig")

def jaccard_sig(a, b):
    aa = set(str(a).split("|"))
    bb = set(str(b).split("|"))
    return len(aa & bb) / len(aa | bb) if (aa | bb) else np.nan

lineups = test_round.pivot_table(
    index=["temporada", "rodada_target"],
    columns="strategy",
    values="lineup_signature",
    aggfunc="first",
).reset_index()

similarity_rows = []
if {"FAME-DOC", "FAME-Fixed 70/20/10"}.issubset(lineups.columns):
    for _, r in lineups.iterrows():
        similarity_rows.append({
            "temporada": r["temporada"],
            "rodada_target": r["rodada_target"],
            "jaccard_doc_vs_fixed": jaccard_sig(r["FAME-DOC"], r["FAME-Fixed 70/20/10"]),
            "identical_doc_vs_fixed": r["FAME-DOC"] == r["FAME-Fixed 70/20/10"],
        })

lineup_similarity = pd.DataFrame(similarity_rows)
lineup_similarity.to_csv(OUT / "kbs_doc_lineup_similarity_doc_vs_fixed_by_round.csv", index=False, encoding="utf-8-sig")

## 13. Deltas rodada a rodada para discussão do ganho operacional

In [ ]:
utility_wide = test_round.pivot_table(
    index=["temporada", "rodada_target"],
    columns="strategy",
    values="operational_utility_no_captain",
    aggfunc="first",
).reset_index()

if "FAME-DOC" in utility_wide.columns:
    for comparator in ["FAME-Fixed 70/20/10", "Pure prediction", "Availability-adjusted prediction"]:
        if comparator in utility_wide.columns:
            safe = comparator.lower().replace(" ", "_").replace("/", "_").replace("-", "_")
            out = utility_wide[["temporada", "rodada_target", "FAME-DOC", comparator]].copy()
            out["delta_doc_minus_comparator"] = out["FAME-DOC"] - out[comparator]
            out["doc_win"] = out["delta_doc_minus_comparator"] > 0
            out["tie"] = out["delta_doc_minus_comparator"] == 0
            out.to_csv(
                OUT / f"kbs_doc_roundwise_delta_vs_{safe}.csv",
                index=False, encoding="utf-8-sig"
            )

utility_wide.to_csv(OUT / "kbs_doc_test_roundwise_utility_wide.csv", index=False, encoding="utf-8-sig")

## 14. Manifesto, dicionário de dados e catálogo de evidências do artigo

Este bloco garante que toda informação produzida pelo notebook seja localizável.
O catálogo relaciona cada arquivo ao tipo de discussão que ele sustenta no
manuscrito KBS.

In [ ]:
# Catálogo conceitual dos arquivos centrais.
evidence_catalog = pd.DataFrame([
    ["kbs_doc_formation_contract.csv", "Method audit", "Exact 4-3-3 composition: 11 field players + one coach"],
    ["kbs_doc_optimization_contract.csv", "Method audit", "Budget, team-size and club-limit implementation contract"],
    ["kbs_doc_code_paper_consistency_audit.csv", "Method audit", "Corrections required for KBS manuscript relative to ESWA wording"],
    ["kbs_doc_calibration_player_components_2024.csv", "Calibration data", "Player-level decision components used before weight selection"],
    ["kbs_doc_calibration_results_by_round_all_weights.csv", "Calibration", "Operational utility of every weight vector in every 2024 round"],
    ["kbs_doc_calibration_selected_players_all_weights.csv", "Calibration", "Exact selected lineup for every weight vector and 2024 round"],
    ["kbs_doc_cal_summary_all_weights.csv", "Calibration", "Global calibration landscape and optimal weights"],
    ["kbs_doc_calibration_near_optimal_region.csv", "Robustness", "Near-optimal calibration region"],
    ["kbs_doc_selected_weights_frozen.csv", "Method", "Weights selected in 2024 and frozen before 2025"],
    ["kbs_doc_test_player_scores_2025.csv", "Independent test", "Player-level Fixed, DOC and baseline scores"],
    ["kbs_doc_test_results_by_round.csv", "Independent test", "Round-level operational utility for all strategies"],
    ["kbs_doc_test_selected_players_by_strategy.csv", "Independent test", "Exact selected players for each strategy and round"],
    ["kbs_doc_test_strategy_summary.csv", "Independent test", "Mean, SD, cumulative utility and gains"],
    ["kbs_doc_test_bootstrap_ci.csv", "Inference", "Bootstrap confidence intervals"],
    ["kbs_doc_test_friedman.csv", "Inference", "Global Friedman test"],
    ["kbs_doc_test_pairwise_wilcoxon_holm.csv", "Inference", "All paired Wilcoxon comparisons with Holm correction"],
    ["kbs_doc_ranking_metrics_by_round.csv", "Ranking", "Precision, Recall, NDCG, Spearman and Kendall by round"],
    ["kbs_doc_ranking_metrics_summary.csv", "Ranking", "Aggregated ranking performance"],
    ["kbs_doc_test_2025_posthoc_sensitivity_summary_all_weights.csv", "Robustness", "Out-of-sample sensitivity landscape, not used for selection"],
    ["kbs_doc_test_2025_posthoc_sensitivity_by_round_all_weights.csv", "Robustness", "Round-level out-of-sample sensitivity"],
    ["kbs_doc_test_2025_posthoc_sensitivity_players_all_weights.csv", "Robustness", "Exact lineups for all post-hoc 2025 weights"],
    ["kbs_doc_temporal_weight_stability.csv", "Temporal stability", "Distance between calibrated and retrospective-optimal weights"],
    ["kbs_doc_lineup_similarity_doc_vs_fixed_by_round.csv", "Decision stability", "Jaccard similarity of DOC and Fixed lineups"],
    ["kbs_doc_test_roundwise_utility_wide.csv", "Operational discussion", "Roundwise utilities in wide form"],
], columns=["file", "section", "supports"])
evidence_catalog.to_csv(OUT / "kbs_doc_evidence_catalog.csv", index=False, encoding="utf-8-sig")

# Inventário de todos os CSVs gerados.
manifest_rows = []
for p in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(p)
    manifest_rows.append({
        "filename": p.name,
        "n_rows": len(df),
        "n_columns": df.shape[1],
        "size_bytes": p.stat().st_size,
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
    })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUT / "kbs_doc_manifest.csv", index=False, encoding="utf-8-sig")

# Dicionário de dados global.
dictionary = []
for p in sorted(OUT.glob("*.csv")):
    if p.name in {"kbs_doc_data_dictionary.csv"}:
        continue
    df = pd.read_csv(p)
    for c in df.columns:
        dictionary.append({
            "filename": p.name,
            "column": c,
            "dtype": str(df[c].dtype),
            "n_missing": int(df[c].isna().sum()),
            "missing_pct": float(df[c].isna().mean() * 100) if len(df) else np.nan,
        })
pd.DataFrame(dictionary).to_csv(
    OUT / "kbs_doc_data_dictionary.csv", index=False, encoding="utf-8-sig"
)

# Configuração computacional/metodológica.
method = pd.DataFrame([
    {"parameter": "calibration_season", "value": 2024},
    {"parameter": "independent_test_season", "value": 2025},
    {"parameter": "calibration_objective", "value": "mean operational utility without captain"},
    {"parameter": "weight_grid_step", "value": GRID_STEP},
    {"parameter": "n_weight_combinations", "value": len(weight_grid)},
    {"parameter": "fixed_weights", "value": str(REFERENCE_WEIGHTS)},
    {"parameter": "doc_weights", "value": str(DOC_WEIGHTS)},
    {"parameter": "posthoc_2025_best_weights_not_used_for_selection", "value": str(TEST_RETROSPECTIVE_BEST)},
    {"parameter": "formation", "value": FORMATION},
    {"parameter": "field_players", "value": EXPECTED_FIELD_PLAYERS},
    {"parameter": "coaches", "value": EXPECTED_COACHES},
    {"parameter": "total_selected_entities", "value": EXPECTED_TOTAL_SELECTED},
    {"parameter": "budget_max", "value": BUDGET_MAX},
    {"parameter": "max_per_club", "value": MAX_PER_CLUB},
    {"parameter": "club_limit_interpretation", "value": "implemented but effectively nonbinding in the main experiment"},
    {"parameter": "require_probable", "value": REQUIRE_PROBABLE},
    {"parameter": "python", "value": sys.version.split()[0]},
    {"parameter": "platform", "value": platform.platform()},
    {"parameter": "seed", "value": SEED},
    {"parameter": "bootstrap_seed", "value": BOOTSTRAP_SEED},
])
method.to_csv(OUT / "kbs_doc_method_configuration.csv", index=False, encoding="utf-8-sig")

print("Pacote completo KBS salvo em:", OUT.resolve())
display(manifest)


## 15. Ablation study da camada prediction-to-decision — teste independente de 2025

Este bloco refaz o estudo de ablação sob o protocolo KBS, sem reutilizar os
resultados do protocolo ESWA.

Princípios do experimento:

1. todas as ablações são avaliadas exclusivamente nas 37 rodadas independentes de 2025;
2. a formação, o orçamento, a elegibilidade e o solver permanecem idênticos;
3. a utilidade operacional é a pontuação observada da escalação **sem bônus de capitão**;
4. FAME-Fixed (70/20/10) é a referência estrutural da ablação;
5. FAME-DOC permanece uma parametrização alternativa, não uma ablação;
6. ao remover um componente ponderado, os pesos remanescentes são renormalizados;
7. “No availability adjustment” remove o ajuste de disponibilidade tanto do
   componente de desempenho esperado quanto do componente de teto, preservando
   variabilidade e eficiência econômica;
8. são exportados resultados por rodada, jogadores selecionados, bootstrap,
   Wilcoxon-Holm, similaridade de escalação e perdas relativas ao FAME-Fixed.

As configurações avaliadas são:

- FAME-Fixed 70/20/10 — referência completa;
- No availability adjustment;
- No ceiling component;
- No economic-efficiency component;
- Pure prediction — remoção completa da transformação multicritério;
- Availability-adjusted only;
- Ceiling only;
- Economic efficiency only.

O objetivo não é demonstrar que uma arquitetura “completa” vence todas as
simplificações, mas identificar quais partes da transformação alteram
materialmente a utilidade e as decisões out-of-time.


In [ ]:
# ============================================================
# 15. ABLATION STUDY — PROTOCOLO KBS / 2025
# ============================================================

def _normalize_weights(*weights):
    arr = np.asarray(weights, dtype=float)
    total = arr.sum()
    if total <= 0:
        raise ValueError("A soma dos pesos remanescentes deve ser positiva.")
    return tuple(arr / total)

ablation_eval = test_eval.copy()

# Variabilidade utilizada para reconstruir o teto SEM disponibilidade.
if "std_shrunk" in ablation_eval.columns:
    _std_for_ablation = pd.to_numeric(ablation_eval["std_shrunk"], errors="coerce").fillna(0.0)
elif "std_5" in ablation_eval.columns:
    _std_for_ablation = pd.to_numeric(ablation_eval["std_5"], errors="coerce").fillna(0.0)
else:
    raise ValueError("A ablação de disponibilidade requer std_shrunk ou std_5.")

# Componentes contrafactuais necessários para isolar disponibilidade.
ablation_eval["expected_no_availability"] = ablation_eval["pred_ensemble"]
ablation_eval["ceiling_no_availability"] = (
    ablation_eval["pred_ensemble"]
    + CEILING_VARIABILITY_FACTOR * _std_for_ablation
)
ablation_eval["component_expected_no_availability_norm"] = minmax_por_rodada(
    ablation_eval, "expected_no_availability"
)
ablation_eval["component_ceiling_no_availability_norm"] = minmax_por_rodada(
    ablation_eval, "ceiling_no_availability"
)

# Referência completa.
ablation_eval["ab_full_fixed"] = ablation_eval["score_fame_fixed"]

# 1) Remove availability from BOTH availability-dependent components.
ablation_eval["ab_no_availability"] = (
    REFERENCE_WEIGHTS[0] * ablation_eval["component_expected_no_availability_norm"]
    + REFERENCE_WEIGHTS[1] * ablation_eval["component_ceiling_no_availability_norm"]
    + REFERENCE_WEIGHTS[2] * ablation_eval["component_economic_norm"]
)

# 2) Remove ceiling contribution and renormalize remaining weights.
w_exp_no_ceil, w_econ_no_ceil = _normalize_weights(
    REFERENCE_WEIGHTS[0], REFERENCE_WEIGHTS[2]
)
ablation_eval["ab_no_ceiling"] = (
    w_exp_no_ceil * ablation_eval["component_expected_norm"]
    + w_econ_no_ceil * ablation_eval["component_economic_norm"]
)

# 3) Remove economic-efficiency contribution and renormalize.
w_exp_no_econ, w_ceil_no_econ = _normalize_weights(
    REFERENCE_WEIGHTS[0], REFERENCE_WEIGHTS[1]
)
ablation_eval["ab_no_economic"] = (
    w_exp_no_econ * ablation_eval["component_expected_norm"]
    + w_ceil_no_econ * ablation_eval["component_ceiling_norm"]
)

# 4) Entire decision-transformation layer removed.
ablation_eval["ab_pure_prediction"] = ablation_eval["pred_ensemble"]

# 5–7) Single-component representations.
ablation_eval["ab_expected_only"] = ablation_eval["pred_ajustada_prob"]
ablation_eval["ab_ceiling_only"] = ablation_eval["teto_score"]
ablation_eval["ab_economic_only"] = ablation_eval["roi"]

ABLATION_STRATEGIES = {
    "FAME-Fixed 70/20/10": {
        "score_col": "ab_full_fixed",
        "type": "reference",
        "removed": "none",
    },
    "No availability adjustment": {
        "score_col": "ab_no_availability",
        "type": "component_removed",
        "removed": "availability adjustment",
    },
    "No ceiling component": {
        "score_col": "ab_no_ceiling",
        "type": "component_removed",
        "removed": "ceiling",
    },
    "No economic-efficiency component": {
        "score_col": "ab_no_economic",
        "type": "component_removed",
        "removed": "economic efficiency",
    },
    "Pure prediction": {
        "score_col": "ab_pure_prediction",
        "type": "layer_removed",
        "removed": "entire multicriteria transformation",
    },
    "Availability-adjusted only": {
        "score_col": "ab_expected_only",
        "type": "single_component",
        "removed": "ceiling + economic efficiency",
    },
    "Ceiling only": {
        "score_col": "ab_ceiling_only",
        "type": "single_component",
        "removed": "expected + economic efficiency",
    },
    "Economic efficiency only": {
        "score_col": "ab_economic_only",
        "type": "single_component",
        "removed": "expected + ceiling",
    },
}

ab_round_parts, ab_player_parts, ab_fail_parts = [], [], []

for strategy_name, cfg in ABLATION_STRATEGIES.items():
    rr, pp, ff = evaluate_score_column(
        ablation_eval,
        cfg["score_col"],
        strategy_name,
        extra_metadata={
            "ablation_type": cfg["type"],
            "removed_component": cfg["removed"],
        },
    )
    ab_round_parts.append(rr)
    ab_player_parts.append(pp)
    if not ff.empty:
        ab_fail_parts.append(ff)

ablation_round = pd.concat(ab_round_parts, ignore_index=True)
ablation_players = pd.concat(ab_player_parts, ignore_index=True)
ablation_failures = (
    pd.concat(ab_fail_parts, ignore_index=True)
    if ab_fail_parts else pd.DataFrame()
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
ablation_summary = (
    ablation_round.groupby(
        ["strategy", "ablation_type", "removed_component"], as_index=False
    )
    .agg(
        n_rounds=("operational_utility_no_captain", "size"),
        mean_utility=("operational_utility_no_captain", "mean"),
        median_utility=("operational_utility_no_captain", "median"),
        sd_utility=("operational_utility_no_captain", "std"),
        cumulative_utility=("operational_utility_no_captain", "sum"),
        min_utility=("operational_utility_no_captain", "min"),
        max_utility=("operational_utility_no_captain", "max"),
        mean_cost=("cost_total", "mean"),
        mean_budget_remaining=("budget_remaining", "mean"),
    )
)

_ref_mean = float(
    ablation_summary.loc[
        ablation_summary["strategy"].eq("FAME-Fixed 70/20/10"),
        "mean_utility",
    ].iloc[0]
)
ablation_summary["delta_mean_vs_fixed"] = (
    ablation_summary["mean_utility"] - _ref_mean
)
ablation_summary["loss_vs_fixed"] = (
    _ref_mean - ablation_summary["mean_utility"]
)
ablation_summary["loss_pct_vs_fixed"] = np.where(
    _ref_mean != 0,
    100.0 * ablation_summary["loss_vs_fixed"] / _ref_mean,
    np.nan,
)
ablation_summary = ablation_summary.sort_values(
    "mean_utility", ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Bootstrap CI
# ------------------------------------------------------------
ab_boot_rows = []
for i, (strategy, g) in enumerate(ablation_round.groupby("strategy")):
    lo, hi = bootstrap_ci(
        g["operational_utility_no_captain"],
        seed=BOOTSTRAP_SEED + 100 + i,
    )
    ab_boot_rows.append({
        "strategy": strategy,
        "mean": g["operational_utility_no_captain"].mean(),
        "ci95_low": lo,
        "ci95_high": hi,
        "n_rounds": len(g),
    })
ablation_bootstrap = pd.DataFrame(ab_boot_rows)

# ------------------------------------------------------------
# Omnibus Friedman across all ablation configurations
# ------------------------------------------------------------
ab_pivot = ablation_round.pivot_table(
    index=["temporada", "rodada_target"],
    columns="strategy",
    values="operational_utility_no_captain",
    aggfunc="first",
).dropna()

ablation_friedman = pd.DataFrame()
if ab_pivot.shape[1] >= 3 and len(ab_pivot) > 0:
    stat, p = friedmanchisquare(*[ab_pivot[c].values for c in ab_pivot.columns])
    ablation_friedman = pd.DataFrame([{
        "test": "Friedman",
        "statistic": stat,
        "p_value": p,
        "n_rounds_complete": len(ab_pivot),
        "n_strategies": ab_pivot.shape[1],
    }])

# ------------------------------------------------------------
# Fixed-reference Wilcoxon tests; Holm correction ONLY over
# the seven planned ablation contrasts.
# ------------------------------------------------------------
AB_REF = "FAME-Fixed 70/20/10"
ab_pair_rows = []

for comparator in [c for c in ab_pivot.columns if c != AB_REF]:
    diff = ab_pivot[AB_REF] - ab_pivot[comparator]
    try:
        stat, p = wilcoxon(
            diff,
            zero_method="wilcox",
            alternative="two-sided",
        )
    except ValueError:
        stat, p = 0.0, 1.0

    ab_pair_rows.append({
        "reference": AB_REF,
        "comparator": comparator,
        "n_rounds": len(diff),
        "mean_reference": ab_pivot[AB_REF].mean(),
        "mean_comparator": ab_pivot[comparator].mean(),
        "mean_diff_fixed_minus_comparator": diff.mean(),
        "median_diff_fixed_minus_comparator": diff.median(),
        "fixed_wins_pct": 100 * (diff > 0).mean(),
        "ties_pct": 100 * (diff == 0).mean(),
        "fixed_losses_pct": 100 * (diff < 0).mean(),
        "wilcoxon_stat": stat,
        "p_raw": p,
    })

ablation_pairwise = pd.DataFrame(ab_pair_rows)
if not ablation_pairwise.empty:
    ablation_pairwise["p_holm"] = holm_adjust(
        ablation_pairwise["p_raw"].to_numpy()
    )
    ablation_pairwise["significant_5pct"] = (
        ablation_pairwise["p_holm"] < 0.05
    )
    ablation_pairwise = ablation_pairwise.sort_values(
        "p_holm"
    ).reset_index(drop=True)

# ------------------------------------------------------------
# Lineup similarity to FAME-Fixed
# ------------------------------------------------------------
ab_lineups = ablation_round.pivot_table(
    index=["temporada", "rodada_target"],
    columns="strategy",
    values="lineup_signature",
    aggfunc="first",
).reset_index()

ab_similarity_rows = []
for comparator in [c for c in ab_lineups.columns if c not in {"temporada", "rodada_target", AB_REF}]:
    for _, row in ab_lineups.iterrows():
        ab_similarity_rows.append({
            "temporada": row["temporada"],
            "rodada_target": row["rodada_target"],
            "reference": AB_REF,
            "comparator": comparator,
            "jaccard_vs_fixed": jaccard_sig(row[AB_REF], row[comparator]),
            "identical_vs_fixed": row[AB_REF] == row[comparator],
        })

ablation_similarity_by_round = pd.DataFrame(ab_similarity_rows)

ablation_similarity_summary = (
    ablation_similarity_by_round.groupby(
        ["reference", "comparator"], as_index=False
    )
    .agg(
        mean_jaccard=("jaccard_vs_fixed", "mean"),
        sd_jaccard=("jaccard_vs_fixed", "std"),
        min_jaccard=("jaccard_vs_fixed", "min"),
        max_jaccard=("jaccard_vs_fixed", "max"),
        identical_rounds=("identical_vs_fixed", "sum"),
        n_rounds=("identical_vs_fixed", "size"),
    )
)
ablation_similarity_summary["identical_pct"] = (
    100.0
    * ablation_similarity_summary["identical_rounds"]
    / ablation_similarity_summary["n_rounds"]
)

# ------------------------------------------------------------
# Round-wise deltas against FAME-Fixed
# ------------------------------------------------------------
ab_delta_rows = []
for comparator in [c for c in ab_pivot.columns if c != AB_REF]:
    d = (ab_pivot[comparator] - ab_pivot[AB_REF]).rename("delta_vs_fixed").reset_index()
    d["comparator"] = comparator
    ab_delta_rows.append(d)
ablation_roundwise_delta = pd.concat(ab_delta_rows, ignore_index=True)

# ------------------------------------------------------------
# Configuration contract / audit table
# ------------------------------------------------------------
ablation_contract = pd.DataFrame([
    {
        "strategy": k,
        "score_column": v["score_col"],
        "ablation_type": v["type"],
        "removed_component": v["removed"],
        "evaluation_season": 2025,
        "captain_bonus_in_utility": False,
        "same_constraints_as_fixed": True,
        "same_candidate_pool_as_fixed": True,
    }
    for k, v in ABLATION_STRATEGIES.items()
])

# ------------------------------------------------------------
# Export ALL evidence required for manuscript discussion
# ------------------------------------------------------------
ablation_eval.to_csv(
    OUT / "kbs_doc_ablation_player_scores_2025.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_round.to_csv(
    OUT / "kbs_doc_ablation_results_by_round.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_players.to_csv(
    OUT / "kbs_doc_ablation_selected_players.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_summary.to_csv(
    OUT / "kbs_doc_ablation_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_bootstrap.to_csv(
    OUT / "kbs_doc_ablation_bootstrap_ci.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_friedman.to_csv(
    OUT / "kbs_doc_ablation_friedman.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_pairwise.to_csv(
    OUT / "kbs_doc_ablation_wilcoxon_vs_fixed_holm.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_similarity_by_round.to_csv(
    OUT / "kbs_doc_ablation_lineup_similarity_by_round.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_similarity_summary.to_csv(
    OUT / "kbs_doc_ablation_lineup_similarity_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_roundwise_delta.to_csv(
    OUT / "kbs_doc_ablation_roundwise_delta_vs_fixed.csv",
    index=False,
    encoding="utf-8-sig",
)
ablation_contract.to_csv(
    OUT / "kbs_doc_ablation_contract.csv",
    index=False,
    encoding="utf-8-sig",
)
if not ablation_failures.empty:
    ablation_failures.to_csv(
        OUT / "kbs_doc_ablation_failures.csv",
        index=False,
        encoding="utf-8-sig",
    )

display(ablation_summary)
display(ablation_friedman)
display(ablation_pairwise)
display(ablation_similarity_summary)


## 16. Figuras essenciais para Results e Ablation Study

O bloco abaixo gera figuras em PNG (300 dpi) e PDF vetorial. As figuras principais
foram escolhidas para responder diretamente às perguntas da versão KBS:

- onde está o ótimo de calibração em 2024?
- o landscape decisório muda em 2025?
- quais estratégias apresentam melhor utilidade out-of-time e com qual incerteza?
- quais componentes da camada prediction-to-decision alteram a utilidade?
- mudanças de componentes alteram efetivamente as escalações?
- qual é a estabilidade rodada a rodada entre FAME-DOC e FAME-Fixed?

As figuras de ablação são produzidas exclusivamente a partir do teste independente
de 2025.


In [ ]:
# ============================================================
# 16. FIGURAS PARA O MANUSCRITO KBS
# ============================================================
import matplotlib.pyplot as plt

# Verificação das dependências produzidas nas células anteriores
_required_objects = [
    "cal_summary",
    "test_sens_summary",
    "test_summary",
    "bootstrap_table",
    "lineup_similarity",
    "ablation_summary",
    "ablation_bootstrap",
    "ablation_similarity_summary",
    "ablation_roundwise_delta",
]
_missing_objects = [name for name in _required_objects if name not in globals()]
if _missing_objects:
    raise RuntimeError(
        "Execute o notebook sequencialmente desde o início. "
        "Objetos ainda não definidos: " + ", ".join(_missing_objects)
    )

FIG_DIR = OUT / "figures_kbs"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def simplex_xy(w_expected, w_ceiling, w_economic):
    x = w_ceiling + 0.5 * w_economic
    y = (np.sqrt(3) / 2.0) * w_economic
    return x, y

def save_current_figure(stem):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()
    plt.close()

# ------------------------------------------------------------
# Figure R1 — 2024 calibration landscape
# ------------------------------------------------------------
cal_plot = cal_summary.copy()
cal_plot["x_simplex"], cal_plot["y_simplex"] = simplex_xy(
    cal_plot["w_expected"].to_numpy(),
    cal_plot["w_ceiling"].to_numpy(),
    cal_plot["w_economic"].to_numpy(),
)

doc_row = cal_plot.loc[cal_plot["rank_mean"].eq(1)].iloc[0]
fixed_mask = (
    np.isclose(cal_plot["w_expected"], REFERENCE_WEIGHTS[0])
    & np.isclose(cal_plot["w_ceiling"], REFERENCE_WEIGHTS[1])
    & np.isclose(cal_plot["w_economic"], REFERENCE_WEIGHTS[2])
)
fixed_cal_row = cal_plot.loc[fixed_mask].iloc[0]

plt.figure(figsize=(8.2, 6.8))
sc = plt.scatter(
    cal_plot["x_simplex"],
    cal_plot["y_simplex"],
    c=cal_plot["mean_utility"],
    s=48,
)
plt.colorbar(sc, label="Mean realized utility (2024)")
plt.scatter(
    [doc_row["x_simplex"]], [doc_row["y_simplex"]],
    marker="*", s=220, label="FAME-DOC selected in 2024",
)
plt.scatter(
    [fixed_cal_row["x_simplex"]], [fixed_cal_row["y_simplex"]],
    marker="X", s=140, label="FAME-Fixed 70/20/10",
)
plt.plot([0, 1, 0.5, 0], [0, 0, np.sqrt(3)/2, 0], linewidth=1)
plt.text(-0.03, -0.04, "Expected", ha="right", va="top")
plt.text(1.03, -0.04, "Ceiling", ha="left", va="top")
plt.text(0.5, np.sqrt(3)/2 + 0.04, "Economic", ha="center", va="bottom")
plt.title("Decision-layer calibration landscape in 2024")
plt.xticks([])
plt.yticks([])
plt.legend(frameon=False)
save_current_figure("fig_results_2024_calibration_landscape")

# ------------------------------------------------------------
# Figure R2 — 2025 post-hoc landscape
# ------------------------------------------------------------
post_plot = test_sens_summary.copy()
post_plot["x_simplex"], post_plot["y_simplex"] = simplex_xy(
    post_plot["w_expected"].to_numpy(),
    post_plot["w_ceiling"].to_numpy(),
    post_plot["w_economic"].to_numpy(),
)

best25_row = post_plot.loc[post_plot["rank_mean"].eq(1)].iloc[0]
doc_mask = (
    np.isclose(post_plot["w_expected"], DOC_WEIGHTS[0])
    & np.isclose(post_plot["w_ceiling"], DOC_WEIGHTS[1])
    & np.isclose(post_plot["w_economic"], DOC_WEIGHTS[2])
)
fixed_mask25 = (
    np.isclose(post_plot["w_expected"], REFERENCE_WEIGHTS[0])
    & np.isclose(post_plot["w_ceiling"], REFERENCE_WEIGHTS[1])
    & np.isclose(post_plot["w_economic"], REFERENCE_WEIGHTS[2])
)

doc25_row = post_plot.loc[doc_mask].iloc[0]
fixed25_row = post_plot.loc[fixed_mask25].iloc[0]

plt.figure(figsize=(8.2, 6.8))
sc = plt.scatter(
    post_plot["x_simplex"],
    post_plot["y_simplex"],
    c=post_plot["mean_utility"],
    s=48,
)
plt.colorbar(sc, label="Mean realized utility (2025, post hoc)")
plt.scatter(
    [best25_row["x_simplex"]], [best25_row["y_simplex"]],
    marker="*", s=220, label="Retrospective best 2025",
)
plt.scatter(
    [fixed25_row["x_simplex"]], [fixed25_row["y_simplex"]],
    marker="X", s=140, label="FAME-Fixed 70/20/10",
)
plt.scatter(
    [doc25_row["x_simplex"]], [doc25_row["y_simplex"]],
    marker="D", s=110, label="FAME-DOC frozen from 2024",
)
plt.plot([0, 1, 0.5, 0], [0, 0, np.sqrt(3)/2, 0], linewidth=1)
plt.text(-0.03, -0.04, "Expected", ha="right", va="top")
plt.text(1.03, -0.04, "Ceiling", ha="left", va="top")
plt.text(0.5, np.sqrt(3)/2 + 0.04, "Economic", ha="center", va="bottom")
plt.title("Post-hoc decision-layer landscape in 2025")
plt.xticks([])
plt.yticks([])
plt.legend(frameon=False)
save_current_figure("fig_results_2025_posthoc_landscape")

# ------------------------------------------------------------
# Figure R3 — Main operational comparison, 2025
# ------------------------------------------------------------
perf = test_summary.merge(
    bootstrap_table[["strategy", "ci95_low", "ci95_high"]],
    on="strategy",
    how="left",
).sort_values("mean_utility", ascending=True)

xerr = np.vstack([
    perf["mean_utility"].to_numpy() - perf["ci95_low"].to_numpy(),
    perf["ci95_high"].to_numpy() - perf["mean_utility"].to_numpy(),
])

plt.figure(figsize=(9.0, 6.4))
plt.errorbar(
    perf["mean_utility"],
    perf["strategy"],
    xerr=xerr,
    fmt="o",
    capsize=3,
)
plt.xlabel("Mean realized lineup utility in 2025")
plt.ylabel("")
plt.title("Out-of-time operational performance with 95% bootstrap confidence intervals")
save_current_figure("fig_results_2025_operational_utility_ci")

# ------------------------------------------------------------
# Figure R4 — DOC vs Fixed lineup stability
# ------------------------------------------------------------
sim_plot = lineup_similarity.sort_values("rodada_target").copy()

plt.figure(figsize=(9.0, 5.2))
plt.plot(
    sim_plot["rodada_target"],
    sim_plot["jaccard_doc_vs_fixed"],
    marker="o",
    linewidth=1.5,
)
plt.axhline(
    sim_plot["jaccard_doc_vs_fixed"].mean(),
    linestyle="--",
    linewidth=1,
    label=f"Mean Jaccard = {sim_plot['jaccard_doc_vs_fixed'].mean():.3f}",
)
plt.ylim(0, 1.05)
plt.xlabel("2025 round")
plt.ylabel("Jaccard similarity")
plt.title("Round-wise lineup similarity: FAME-DOC vs FAME-Fixed")
plt.legend(frameon=False)
save_current_figure("fig_results_2025_lineup_similarity")

# ------------------------------------------------------------
# Figure A1 — Ablation operational utility + bootstrap CI
# ESSENTIAL for main manuscript
# ------------------------------------------------------------
ab_perf = ablation_summary.merge(
    ablation_bootstrap[["strategy", "ci95_low", "ci95_high"]],
    on="strategy",
    how="left",
).sort_values("mean_utility", ascending=True)

ab_xerr = np.vstack([
    ab_perf["mean_utility"].to_numpy() - ab_perf["ci95_low"].to_numpy(),
    ab_perf["ci95_high"].to_numpy() - ab_perf["mean_utility"].to_numpy(),
])

plt.figure(figsize=(9.4, 6.8))
plt.errorbar(
    ab_perf["mean_utility"],
    ab_perf["strategy"],
    xerr=ab_xerr,
    fmt="o",
    capsize=3,
)
plt.axvline(
    _ref_mean,
    linestyle="--",
    linewidth=1,
    label="FAME-Fixed mean",
)
plt.xlabel("Mean realized lineup utility in 2025")
plt.ylabel("")
plt.title("Ablation study of the prediction-to-decision layer")
plt.legend(frameon=False)
save_current_figure("fig_results_ablation_utility_ci")

# ------------------------------------------------------------
# Figure A2 — Practical decision impact:
# mean Jaccard vs loss in utility
# ESSENTIAL for main manuscript
# ------------------------------------------------------------
ab_impact = (
    ablation_summary.loc[
        ~ablation_summary["strategy"].eq(AB_REF),
        ["strategy", "loss_vs_fixed", "delta_mean_vs_fixed"]
    ]
    .merge(
        ablation_similarity_summary[
            ["comparator", "mean_jaccard", "identical_pct"]
        ],
        left_on="strategy",
        right_on="comparator",
        how="left",
    )
)

plt.figure(figsize=(8.8, 6.2))
plt.scatter(
    ab_impact["mean_jaccard"],
    ab_impact["loss_vs_fixed"],
    s=70,
)
for _, r in ab_impact.iterrows():
    plt.annotate(
        r["strategy"],
        (r["mean_jaccard"], r["loss_vs_fixed"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )
plt.axhline(0, linewidth=1)
plt.xlabel("Mean lineup Jaccard similarity vs FAME-Fixed")
plt.ylabel("Mean utility loss vs FAME-Fixed")
plt.title("Decision impact of prediction-to-decision ablations")
save_current_figure("fig_results_ablation_decision_impact")

# ------------------------------------------------------------
# Figure A3 — Roundwise delta distribution
# Useful as supplementary/robustness figure
# ------------------------------------------------------------
delta_order = (
    ablation_roundwise_delta.groupby("comparator")["delta_vs_fixed"]
    .mean()
    .sort_values()
    .index
    .tolist()
)
delta_data = [
    ablation_roundwise_delta.loc[
        ablation_roundwise_delta["comparator"].eq(s),
        "delta_vs_fixed",
    ].to_numpy()
    for s in delta_order
]

plt.figure(figsize=(10.0, 6.4))
plt.boxplot(delta_data, vert=False, tick_labels=delta_order)
plt.axvline(0, linewidth=1)
plt.xlabel("Round-level utility difference: ablation minus FAME-Fixed")
plt.ylabel("")
plt.title("Round-wise operational effects of ablations")
save_current_figure("fig_results_ablation_roundwise_delta")

figure_catalog = pd.DataFrame([
    {
        "figure_file": "fig_results_2024_calibration_landscape.pdf",
        "status": "main",
        "role_in_paper": "2024 DOC calibration landscape and selected/fixed parameter locations",
    },
    {
        "figure_file": "fig_results_2025_posthoc_landscape.pdf",
        "status": "main",
        "role_in_paper": "Post-hoc 2025 decision landscape and temporal displacement",
    },
    {
        "figure_file": "fig_results_2025_operational_utility_ci.pdf",
        "status": "main",
        "role_in_paper": "Out-of-time operational comparison with bootstrap uncertainty",
    },
    {
        "figure_file": "fig_results_2025_lineup_similarity.pdf",
        "status": "main_or_supplement",
        "role_in_paper": "Round-wise operational consequence of DOC vs Fixed parameter changes",
    },
    {
        "figure_file": "fig_results_ablation_utility_ci.pdf",
        "status": "main",
        "role_in_paper": "Ablation effects on operational utility with uncertainty",
    },
    {
        "figure_file": "fig_results_ablation_decision_impact.pdf",
        "status": "main",
        "role_in_paper": "Trade-off between lineup change and utility loss under component ablations",
    },
    {
        "figure_file": "fig_results_ablation_roundwise_delta.pdf",
        "status": "supplement",
        "role_in_paper": "Distribution of round-wise utility effects relative to FAME-Fixed",
    },
])
figure_catalog.to_csv(
    OUT / "kbs_doc_results_figure_catalog.csv",
    index=False,
    encoding="utf-8-sig",
)
display(figure_catalog)


## 17. Atualização final da rastreabilidade


In [ ]:
# ============================================================
# 17. ATUALIZAÇÃO FINAL DO CATÁLOGO, MANIFESTO E DICIONÁRIO
# ============================================================
# Acrescenta explicitamente os novos arquivos de ablação ao catálogo de evidências.
ablation_evidence = pd.DataFrame([
    ["kbs_doc_ablation_contract.csv", "Ablation method", "Definition and audit contract for every KBS ablation"],
    ["kbs_doc_ablation_player_scores_2025.csv", "Ablation", "Player-level scores required to reproduce each ablation"],
    ["kbs_doc_ablation_results_by_round.csv", "Ablation", "Round-level operational utility of every ablation"],
    ["kbs_doc_ablation_selected_players.csv", "Ablation", "Exact selected lineups for every ablation and round"],
    ["kbs_doc_ablation_summary.csv", "Ablation", "Descriptive operational summary and losses vs FAME-Fixed"],
    ["kbs_doc_ablation_bootstrap_ci.csv", "Ablation inference", "95% bootstrap confidence intervals"],
    ["kbs_doc_ablation_friedman.csv", "Ablation inference", "Omnibus matched-round Friedman test"],
    ["kbs_doc_ablation_wilcoxon_vs_fixed_holm.csv", "Ablation inference", "Planned Fixed-reference Wilcoxon tests with Holm correction"],
    ["kbs_doc_ablation_lineup_similarity_by_round.csv", "Ablation decisions", "Round-wise Jaccard similarity to FAME-Fixed"],
    ["kbs_doc_ablation_lineup_similarity_summary.csv", "Ablation decisions", "Mean Jaccard and identical-lineup frequencies"],
    ["kbs_doc_ablation_roundwise_delta_vs_fixed.csv", "Ablation robustness", "Round-level operational deltas relative to FAME-Fixed"],
    ["kbs_doc_results_figure_catalog.csv", "Figures", "Traceability between manuscript figures and evidential role"],
], columns=["file", "section", "supports"])

evidence_catalog_final = pd.concat(
    [evidence_catalog, ablation_evidence],
    ignore_index=True,
).drop_duplicates(subset=["file"], keep="last")

evidence_catalog_final.to_csv(
    OUT / "kbs_doc_evidence_catalog.csv",
    index=False,
    encoding="utf-8-sig",
)

# Rebuild manifest after all outputs have been created.
manifest_rows = []
for p in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(p)
    manifest_rows.append({
        "filename": p.name,
        "n_rows": len(df),
        "n_columns": df.shape[1],
        "size_bytes": p.stat().st_size,
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
    })

manifest_final = pd.DataFrame(manifest_rows)
manifest_final.to_csv(
    OUT / "kbs_doc_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

# Rebuild global data dictionary.
dictionary = []
for p in sorted(OUT.glob("*.csv")):
    if p.name == "kbs_doc_data_dictionary.csv":
        continue
    df = pd.read_csv(p)
    for c in df.columns:
        dictionary.append({
            "filename": p.name,
            "column": c,
            "dtype": str(df[c].dtype),
            "n_missing": int(df[c].isna().sum()),
            "missing_pct": float(df[c].isna().mean() * 100) if len(df) else np.nan,
        })

data_dictionary_final = pd.DataFrame(dictionary)
data_dictionary_final.to_csv(
    OUT / "kbs_doc_data_dictionary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(evidence_catalog_final.tail(20))
display(manifest_final.tail(20))
